
# 数据清洗一体化 Notebook（含欧氏距离 KNN 填补 GRP）

本 Notebook 按你的澄清要求完成以下内容：
- 读取并将 `"."`、空字符串、`NA/N/A` 等视为缺失；
- 将 **GRP** 转为数值；
- 使用 **欧几里得距离 KNN**（`KNeighborsRegressor`）**仅对 GRP 进行插补**；
- 计算距离时**排除 `Nweek`** 及非数值列，仅使用与 GRP 更相关的数值特征；
- 为避免量纲影响：对用于计算距离的特征先**中位数填补（仅用于特征缺失）+ 标准化**，不改变原始数据；
- 保存清洗后的 CSV。


In [1]:

# --- 配置 ---
DATA_PATH = "/Users/ying/Library/Mobile Documents/com~apple~CloudDocs/Documents/MasseyUni/MasseyUni_Learning_Business Analytics/156762 Return on Marketing Investment/assignment/assign3-group/762 - A3 Data.csv"      # 请按需修改
OUTPUT_PATH = "/Users/ying/Library/Mobile Documents/com~apple~CloudDocs/Documents/MasseyUni/MasseyUni_Learning_Business Analytics/156762 Return on Marketing Investment/assignment/assign3-group/762 - A3 Data_cleaned.csv"

# 将以下标记视为缺失
NA_MARKERS = [".", " ", "NA", "N/A", "na", "n/a", None]

# --- 导入 ---
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print("Pandas:", pd.__version__)


Pandas: 2.3.1


In [10]:

# --- 读取数据，将特殊标记视为缺失 ---
df = pd.read_csv(DATA_PATH, na_values=NA_MARKERS, keep_default_na=True)
print("原始数据形状:", df.shape)
df.describe()
display(df)

原始数据形状: (156, 12)


,Nweek,TVOL,BVOL,NPP,PRPR,GRP,FTX,DISX,FTDISX,PRCUTX,COUPX,Inflation
0,1,7910828,7183069,1.87,1.45,480.0,36.635513,24.379817,7.854366,30.522735,4.447438,0.001365
1,2,8427629,7208244,1.87,1.50,300.0,36.848491,29.438996,11.085716,28.620498,9.657735,NaN
2,3,8395214,7257232,1.88,1.52,95.0,38.701352,31.873997,6.418356,31.813458,4.669449,NaN
3,4,8023386,7268262,1.88,1.59,56.0,36.791283,34.286075,9.536328,31.480898,4.947495,NaN
4,5,8965263,7603204,1.87,1.53,293.0,42.490380,33.264492,10.550558,33.939781,8.361638,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
151,152,8236845,7058515,2.03,1.76,NaN,47.691722,17.933390,8.951699,37.415413,12.946622,NaN
152,153,8196452,7124040,2.05,1.69,NaN,43.604707,19.468965,8.292255,34.903714,11.200482,NaN
153,154,8368566,7225245,2.05,1.62,NaN,40.037475,22.492238,9.429616,31.455070,9.323835,NaN
154,155,8252303,7274799,2.07,1.74,NaN,40.579571,18.836743,8.315138,33.546527,8.755036,NaN


In [11]:

# --- 结构与缺失情况 ---
display(df.info())

missing_summary = df.isna().sum().sort_values(ascending=False).to_frame("missing_cnt")
missing_summary["missing_pct"] = (missing_summary["missing_cnt"] / len(df)).round(4)
display(missing_summary.head(20))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Nweek      156 non-null    int64  
 1   TVOL       156 non-null    int64  
 2   BVOL       156 non-null    int64  
 3   NPP        156 non-null    float64
 4   PRPR       156 non-null    float64
 5   GRP        144 non-null    float64
 6   FTX        156 non-null    float64
 7   DISX       156 non-null    float64
 8   FTDISX     156 non-null    float64
 9   PRCUTX     156 non-null    float64
 10  COUPX      156 non-null    float64
 11  Inflation  1 non-null      float64
dtypes: float64(9), int64(3)
memory usage: 14.8 KB


None

,missing_cnt,missing_pct
Inflation,155,0.9936
GRP,12,0.0769
Nweek,0,0.0000
TVOL,0,0.0000
NPP,0,0.0000
BVOL,0,0.0000
FTX,0,0.0000
PRPR,0,0.0000
DISX,0,0.0000
FTDISX,0,0.0000



## 将 GRP 转为数值
若包含 `"."` 或其他字符，此步会转为 `NaN`。


In [12]:

if "GRP" not in df.columns:
    raise KeyError("未找到列 'GRP'，请检查列名是否正确。")

# 转为数值（无法解析的转为 NaN）
df["GRP"] = pd.to_numeric(df["GRP"], errors="coerce")

print("GRP 缺失数（转换后）:", int(df["GRP"].isna().sum()))
display(df["GRP"].describe())


GRP 缺失数（转换后）: 12


count    144.000000
mean     262.055556
std      205.630525
min        0.000000
25%       75.250000
50%      232.500000
75%      388.250000
max      926.000000
Name: GRP, dtype: float64


## 选择用于计算欧氏距离的数值特征（排除 Nweek 和 GRP）
只在**特征空间**中进行中位数填补与标准化，**不改写原始数据**；最终仅对 GRP 缺失值进行回填。


In [14]:

# 数值列
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

# 排除列表（可按需扩展）
exclude_cols = ["GRP", "Nweek","Inflation"]
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

if len(feature_cols) == 0:
    raise ValueError("没有可用于欧氏距离的数值特征列（排除 GRP 与 Nweek 后为空）。请确保存在其他数值列。")

print("用于欧氏距离的特征列:")
print(feature_cols)


用于欧氏距离的特征列:
['TVOL', 'BVOL', 'NPP', 'PRPR', 'FTX', 'DISX', 'FTDISX', 'PRCUTX', 'COUPX']



## 使用欧氏距离 KNN 对 GRP 进行插补（仅填补 GRP）
步骤：
1. 特征矩阵 `X = df[feature_cols]`；
2. 用**中位数**填补 `X` 中的缺失（只是为了能计算距离，**不影响原表**）；
3. 对 `X` 做标准化（z-score），避免量纲影响欧氏距离；
4. 在 `y = df['GRP']` 非缺失样本上训练 `KNeighborsRegressor(metric='euclidean', weights='distance')`；
5. 对 `y` 缺失样本进行预测并回填到 `df['GRP']`。


In [15]:

# --- 欧氏距离 KNN 回归，仅插补 GRP ---
num_na_before = int(df["GRP"].isna().sum())
print("GRP 缺失数（插补前）:", num_na_before)

if num_na_before > 0:
    X = df[feature_cols].copy()
    y = df["GRP"].copy()

    # 仅在特征空间中进行：中位数填补 + 标准化
    feat_imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_imputed = feat_imputer.fit_transform(X)   # 不改写 df
    X_scaled = scaler.fit_transform(X_imputed)

    # 训练 / 预测掩码
    mask_train = ~y.isna()
    mask_pred  = y.isna()

    # 训练 KNN 回归器（欧氏距离 + 距离加权）
    knn = KNeighborsRegressor(n_neighbors=5, weights="distance", metric="euclidean")
    knn.fit(X_scaled[mask_train], y[mask_train])

    # 预测缺失的 GRP 并回填
    y_hat = knn.predict(X_scaled[mask_pred])
    df.loc[mask_pred, "GRP"] = y_hat

    print(f"KNN(欧氏) 已插补 GRP。缺失数: {num_na_before} -> {int(df['GRP'].isna().sum())}")
else:
    print("GRP 无缺失，不需插补。")


GRP 缺失数（插补前）: 12
KNN(欧氏) 已插补 GRP。缺失数: 12 -> 0



> 参考：如需切换回**中位数填补**（基线法），可取消下方代码注释。


In [ ]:

# # --- 备选：中位数填补（基线法） ---
# if df["GRP"].isna().sum() > 0:
#     grp_median = df["GRP"].median()
#     df["GRP"] = df["GRP"].fillna(grp_median)
#     print("已使用中位数填补 GRP。")
# else:
#     print("GRP 无缺失，不需中位数填补。")



## 清洗后检查


In [16]:

print("清洗后缺失（前20列）：")
missing_after = df.isna().sum().sort_values(ascending=False).head(20)
display(missing_after)

print("GRP 统计量（插补后）：")
display(df["GRP"].describe())

# 简单分布感知（如需更详细可作图）
print("GRP 前 10 行：")
display(df["GRP"].head(10))


清洗后缺失（前20列）：


Inflation    155
Nweek          0
BVOL           0
TVOL           0
NPP            0
PRPR           0
FTX            0
GRP            0
DISX           0
FTDISX         0
PRCUTX         0
COUPX          0
dtype: int64

GRP 统计量（插补后）：


count    156.000000
mean     248.026611
std      204.075563
min        0.000000
25%       65.000000
50%      215.000000
75%      376.000000
max      926.000000
Name: GRP, dtype: float64

GRP 前 10 行：


0    480.0
1    300.0
2     95.0
3     56.0
4    293.0
5    290.0
6     77.0
7      8.0
8     39.0
9    214.0
Name: GRP, dtype: float64


## 保存清洗结果


In [21]:
df["GRP"] = pd.to_numeric(df["GRP"], errors="coerce").round(0).astype("Int64")
df.to_csv(OUTPUT_PATH, index=False, float_format="%.6f")
print(f"已保存清洗后的数据到: {OUTPUT_PATH}")


已保存清洗后的数据到: /Users/ying/Library/Mobile Documents/com~apple~CloudDocs/Documents/MasseyUni/MasseyUni_Learning_Business Analytics/156762 Return on Marketing Investment/assignment/assign3-group/762 - A3 Data_cleaned.csv
